#### # 1. Initialize Libraries and Window Specification

In [0]:
from pyspark.sql import Window
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# Define window partition rule to rank records per customer
customer_window = Window.partitionBy("cst_id").orderBy(F.col("cst_create_date").desc())

#### 2. Extract Data and Remove Missing Records

In [0]:
# Load raw dataset from bronze catalog and filter out null IDs
raw_df = spark.table("workspace.bronze.crm_cust_info").filter(F.col("cst_id").isNotNull())

#### 3. Deduplicate (Isolate Latest Row per Customer)

In [0]:
# Calculate row numbers and retain only the most recent entry
deduplicated_df = (
    raw_df
    .withColumn("flag_last", F.row_number().over(customer_window))
    .filter(F.col("flag_last") == 1)
    .drop("flag_last")
)

#### # 4. Cleanse Whitespaces Dynamically

In [0]:
# Automatically sweep and trim trailing spaces across all text fields
trimmed_df = deduplicated_df

for field in trimmed_df.schema.fields:
    if isinstance(field.dataType, StringType):
        trimmed_df = trimmed_df.withColumn(field.name, F.trim(F.col(field.name)))

####  # 5. Normalize Categorical Descriptions (Marital Status & Gender)

In [0]:
# Map abbreviation codes into full descriptions
standardized_df = (
    trimmed_df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("N/A")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("N/A")
    )
)

#### # 6. Apply Schema Renaming Map

In [0]:
# Convert raw abbreviations into production column names
renamed_df = standardized_df

RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

for old_name, new_name in RENAME_MAP.items():
    renamed_df = renamed_df.withColumnRenamed(old_name, new_name)

#### # 7. Restructure Final Naming Schema Map and Save to Silver Table

In [0]:
# Select target business columns explicitly using new names and write to target
silver_crm_cust_info_df = renamed_df.select(
    "customer_id",
    "customer_number",
    "first_name",
    "last_name",
    "marital_status",
    "gender",
    "created_date"
)

(
    silver_crm_cust_info_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.crm_customers")
)

silver_crm_cust_info_df.limit(10).display()